In [1]:
import os, sys
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
# Cell 1 (Config)
EMBED_PATH = "../results/embeddings_stratified.npy"
META_PATH  = "../results/metadata_stratified.csv"

emb = np.load(EMBED_PATH)           # shape (n_songs, 768)
meta = pd.read_csv(META_PATH)       # contains the `tag` column

print("Embeddings:", emb.shape)
print("Metadata:", meta.shape)

Embeddings: (30000, 768)
Metadata: (30000, 1)


In [7]:
emb  = np.load(EMBED_PATH)
meta = pd.read_csv(META_PATH)
print(meta.tag.value_counts())
# should be 5000 for each of ['rap','rb','rock','pop','misc','country']

tag
country    5000
misc       5000
pop        5000
rap        5000
rb         5000
rock       5000
Name: count, dtype: int64


In [3]:
# get indices per genre
genre_to_idx = {g: meta.index[meta.tag == g].tolist()
                for g in meta.tag.unique()}

def sample_pair_sims(idxs, other_idxs, n_samples=1000):
    import random
    pairs = [(random.choice(idxs), random.choice(idxs)) for _ in range(n_samples)]
    cross = [(random.choice(idxs), random.choice(other_idxs)) for _ in range(n_samples)]
    sims_within = [np.dot(emb[i], emb[j]) /
                   (np.linalg.norm(emb[i]) * np.linalg.norm(emb[j]))
                   for i,j in pairs]
    sims_across = [np.dot(emb[i], emb[j]) /
                   (np.linalg.norm(emb[i]) * np.linalg.norm(emb[j]))
                   for i,j in cross]
    return sims_within, sims_across

# Example for one genre:
g = list(genre_to_idx.keys())[0]
within, across = sample_pair_sims(genre_to_idx[g],
                                  sum([v for k,v in genre_to_idx.items() if k!=g], []))
print(f"{g}: within mean={np.mean(within):.3f}, across mean={np.mean(across):.3f}")

country: within mean=0.390, across mean=0.331


In [4]:
# K-Means with k = number of genres
k = meta.tag.nunique()
km = KMeans(n_clusters=k, random_state=42).fit(emb)
labels = km.labels_
ari = adjusted_rand_score(meta.tag, labels)
sil = silhouette_score(emb, labels, metric="cosine")
print(f"KMeans ARI: {ari:.3f}, Silhouette: {sil:.3f}")

KMeans ARI: 0.164, Silhouette: 0.057


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    emb, meta.tag, test_size=0.2, random_state=42, stratify=meta.tag)

# k-NN
knn = KNeighborsClassifier(n_neighbors=5)
acc_knn = cross_val_score(knn, X_train, y_train, cv=5, scoring="accuracy").mean()

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
acc_lr = cross_val_score(lr, X_train, y_train, cv=5, scoring="accuracy").mean()

print(f"k-NN CV Acc: {acc_knn:.3f}")
print(f"Logistic CV Acc: {acc_lr:.3f}")

k-NN CV Acc: 0.517
Logistic CV Acc: 0.593


In [6]:
results = {
    "genre": list(genre_to_idx.keys()),
    "within_mean": [...],
    "across_mean": [...],
    "kmeans_ari": ari,
    "kmeans_silhouette": sil,
    "knn_acc": acc_knn,
    "lr_acc": acc_lr
}
pd.DataFrame([results]).to_csv("../results/analysis_summary.csv", index=False)